<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/document-intelligence/lessons/P02-L09-cost-routing-budget/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/document-intelligence/lessons/P02-L09-cost-routing-budget/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/document-intelligence/lessons/P02-L09-cost-routing-budget/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/document-intelligence/lessons/P02-L09-cost-routing-budget/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P02-L09 · Cost per document, and the routing budget that sets it

**You will build:** the budget model for a three-tier document router — a cheap extractor, an
expensive extractor and a human reviewer — solved exactly by sweeping every routing plan and
approximately by the greedy marginal-yield rule, the Pareto frontier of field accuracy against
cost per document, and the one-paragraph recommendation a finance partner would sign.

**Time:** ~75 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download, no model
API · **Prerequisites:** T00-L01 (the 8 GB track), P02-L01 (the evaluation harness), P02-L03
(the tagger), P02-L06 (reviewers who are not perfect), P02-L07 (slices of the traffic).

Every earlier module measured one thing: what an extractor gets right, what a trained model
adds, how fast a reviewer works and how often they are still wrong, which slices of the
traffic are hard. None of them said what to *buy*. This module does. It prices each tier on
each slice of the traffic and decides, under a cap on cost per document, which tier handles
which slice — and then writes the decision down in a form somebody can sign.

**Every price in this notebook is an illustrative placeholder.** None is the price of any real
product, vendor or payroll. Each one is named in the setup cell so that you can replace it with
your own, and every number downstream is recomputed when you do.

By the end you will be able to:

1. Implement cost per document and field accuracy of a routing plan as volume-weighted means,
   and measure how far the mean over segments misstates both.
2. Implement the budget allocation as an exhaustive sweep over every plan, and the Pareto
   frontier of field accuracy against cost per document.
3. Implement the greedy marginal-yield rule, and measure where it matches the sweep and where
   it falls short.
4. Construct a segment mix on which the greedy rule is beaten, and explain why from the
   construction.
5. Generate, from the numbers alone, the routing recommendation a finance partner would sign.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import itertools
import math
import random
import re
import sys
import textwrap
import time
import traceback
from typing import Callable, NamedTuple, Sequence

import numpy as np

import matplotlib
_INTERACTIVE = "ipykernel" in sys.modules
if not _INTERACTIVE:
    # Headless: a script run (including this repository's execution gate) must never try to
    # open a window. In Jupyter the default inline backend is already the right one.
    matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402  (backend must be chosen before this import)

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__,
      "· matplotlib", matplotlib.__version__)

# ---- The price list. EVERY figure in this block is an illustrative placeholder. ----------------
# Carried from module 1 under module 1's own names, so the two modules price a document alike.
MACHINE_COST_PER_DOCUMENT = 0.004   # module 1: the cheap extractor's bill divided by its volume
REVIEWER_COST_PER_HOUR = 30.0       # module 1: a reviewer's loaded cost, in your own currency
# Carried from module 6: the pace its chosen rota ran at, fields an hour at the desk with breaks.
# A simulation's figure, not a benchmark. Time your own reviewers.
REVIEWED_FIELDS_PER_HOUR = 35.6
# New here: a larger extractor billed by the page, because bills for bigger models usually grow
# with the text they read. It is not the price of any real product.
EXPENSIVE_COST_PER_PAGE = 0.03
# The cap finance has set on cost per document. Section 8 writes its recommendation for whatever
# number you put here.
BUDGET_PER_DOCUMENT = 0.30
# ------------------------------------------------------------------------------------------------

# Module 1's schema annotates every document for the same six fields; a wrong field is the unit
# of harm throughout this programme.
FIELDS_PER_DOCUMENT = 6
# Float slack on the cap. Two correct programs can add the same costs in a different order and
# disagree in the last binary digit; a plan that costs exactly the cap must fit in both.
BUDGET_TOLERANCE = 1e-9
# The seed for the random segment mixes in section 6. Nothing else in the lesson is random.
SEED = 20260923


class Tier(NamedTuple):
    """One way to process a document, and what it costs."""
    name: str
    per_document: float     # paid for every document, whatever its length
    per_page: float         # plus this much for every page
    mirrors: str            # which earlier module this tier stands in for


class Segment(NamedTuple):
    """One slice of the monthly traffic, cut on attributes the router can see before reading."""
    name: str
    volume: int                    # documents a month
    pages: int                     # pages per document
    accuracy: tuple[float, ...]    # field accuracy on each tier, in the order of TIERS
    mirrors: str                   # which earlier module's finding the segment is shaped on


class PlanPoint(NamedTuple):
    """Where one routing plan lands: what a document costs, and what share of its fields is right."""
    cost_per_document: float
    accuracy: float


class Sweep(NamedTuple):
    """Every routing plan, and where each one lands."""
    plans: np.ndarray       # int, shape (number of plans, number of segments)
    cost: np.ndarray        # float, cost per document of each plan
    accuracy: np.ndarray    # float, field accuracy of each plan


class Greedy(NamedTuple):
    """The greedy rule's answer, where it started, and the moves it made, in order."""
    plan: tuple[int, ...]
    start: tuple[int, ...]
    moves: tuple[tuple[int, int], ...]   # (segment index, tier index it moved to)


class Recommendation(NamedTuple):
    """Every figure the finance partner's paragraph quotes, computed."""
    budget: float
    plan: tuple[int, ...]
    point: PlanPoint
    plans_checked: int
    monthly_spend: float
    wrong_fields_per_month: float
    greedy_wrong_fields_per_month: float
    next_plan: tuple[int, ...] | None
    next_point: PlanPoint | None
    price_per_wrong_field_avoided: float | None


def unit_cost(segment: Segment, tier: Tier) -> float:
    """What one document of `segment` costs on `tier`: the flat part plus the per-page part."""
    return tier.per_document + tier.per_page * segment.pages


def fits(cost, budget: float):
    """True where a cost per document is at or under the cap. Works on numbers and on arrays."""
    return cost <= budget + BUDGET_TOLERANCE


_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("evaluate_plan",),
    "exercise 2": ("sweep_plans", "best_within_budget"),
    "exercise 3": ("pareto_frontier",),
    "exercise 4": ("greedy_allocation",),
    "exercise 5": ("greedy_counterexample",),
    "exercise 6": ("recommend",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (pareto_frontier)"; several -> "exercises 2, 3 and 4"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _show(fig: "matplotlib.figure.Figure") -> None:
    """Display a figure in Jupyter, or close it cleanly in a headless script run."""
    if _INTERACTIVE:
        plt.show()
    else:
        plt.close(fig)

## 1. The router and its three tiers

A router looks at a document before anything reads it — its vendor, its layout family, its
page count, the quality of its scan, the attributes module 7 sliced by — and sends it to one
of three tiers. The **cheap** tier is module 1's stand-in extractor. The **expensive** tier is
a trained model in the mould of module 3's tagger, billed by the page. The **human** tier is
module 6's reviewer correcting the cheap tier's draft, every field of it, at the pace module
6's chosen rota ran at — so a human-reviewed document pays for its draft as well as for the
reviewer.

Paying for the dearer model only on the inputs where it earns its price is an established
pattern: FrugalGPT, for instance, learns which combinations of language-model APIs to use for
different queries, to cut cost and raise accuracy together, in a market where per-query fees
differ by orders of magnitude (claims.yaml). Run the cell to turn the price list into tiers.

In [ ]:
def build_tiers(fields_per_hour: float = REVIEWED_FIELDS_PER_HOUR) -> tuple[Tier, ...]:
    """The three tiers, priced from the placeholders in the setup cell."""
    review = FIELDS_PER_DOCUMENT * REVIEWER_COST_PER_HOUR / fields_per_hour
    return (
        Tier("cheap", MACHINE_COST_PER_DOCUMENT, 0.0, "module 1's stand-in extractor"),
        Tier("expensive", 0.0, EXPENSIVE_COST_PER_PAGE,
             "a trained model like module 3's tagger, billed by the page"),
        Tier("human", MACHINE_COST_PER_DOCUMENT + review, 0.0,
             "module 6's reviewer correcting the cheap draft, every field"),
    )


TIERS = build_tiers()
print(f"{'tier':11s}{'per document':>13s}{'per page':>10s}   stands in for")
for _t in TIERS:
    print(f"{_t.name:11s}{_t.per_document:13.4f}{_t.per_page:10.4f}   {_t.mirrors}")
print(f"\nthe human line: {MACHINE_COST_PER_DOCUMENT} for the cheap draft + "
      f"{FIELDS_PER_DOCUMENT} fields × {REVIEWER_COST_PER_HOUR:.2f} an hour ÷ "
      f"{REVIEWED_FIELDS_PER_HOUR} fields an hour = {TIERS[2].per_document:.4f} per document")
_one_page = Segment("one page", 1, 1, (), "")
print(f"a human-reviewed one-page document costs "
      f"{unit_cost(_one_page, TIERS[2]) / unit_cost(_one_page, TIERS[0]):,.0f} times a cheap one "
      f"and {unit_cost(_one_page, TIERS[2]) / unit_cost(_one_page, TIERS[1]):,.0f} times an "
      "expensive one.")

## 2. The traffic, sliced

A router cannot price a document it has not read, so it prices **segments**: slices of the
monthly traffic that share the attributes it can see. Each segment below carries its monthly
volume, its page count and a **field accuracy** on each tier — the share of (document, field)
cells right under module 1's normalised matching, true negatives included, which is the unit
module 6 counted its cells still wrong in.

These are fixtures, not measurements: small fixed numbers shaped on what the earlier modules
measured, and the last column says which module each one mirrors. Two of them are traps worth
spotting before the exercises find them for you: one segment on which the expensive tier is
*less* accurate than the cheap one, and one on which it adds nothing at all.

In [ ]:
SEGMENTS: tuple[Segment, ...] = (
    #        name                     volume  pages  accuracy: cheap, expensive, human
    Segment("clean single-page",      30_000,  1, (0.810, 0.964, 0.969),
            "modules 1, 3 and 6 on their own corpora"),
    Segment("two-column layout",       6_000,  1, (0.604, 0.940, 0.969),
            "module 2: raster reading order scrambles fields"),
    Segment("statements with tables",  5_000,  3, (0.550, 0.905, 0.960),
            "module 4: structure errors that content scores miss"),
    Segment("poor scans",              2_500,  1, (0.620, 0.620, 0.930),
            "module 7's scan-quality slice: both models read the same OCR"),
    Segment("new vendor template",     1_500,  1, (0.520, 0.915, 0.969),
            "module 8: a template change the cheap rules never saw"),
    Segment("long supplier names",     4_000,  1, (0.845, 0.838, 0.969),
            "module 3: the tagger cut long unseen names short"),
    Segment("multi-page contracts",    3_000, 12, (0.700, 0.935, 0.950),
            "module 7's page-count slice"),
    Segment("unfamiliar language",       600,  2, (0.400, 0.420, 0.900),
            "module 7's language slice"),
)
TOTAL_VOLUME = sum(s.volume for s in SEGMENTS)

print(f"{'segment':24s}{'docs/month':>11s}{'pages':>6s}   " + "".join(
    f"{t.name + ' acc':>15s}" for t in TIERS) + "   " + "".join(f"{t.name + ' cost':>15s}" for t in TIERS))
for _s in SEGMENTS:
    print(f"{_s.name:24s}{_s.volume:11,d}{_s.pages:6d}   "
          + "".join(f"{a:15.3f}" for a in _s.accuracy) + "   "
          + "".join(f"{unit_cost(_s, t):15.4f}" for t in TIERS))
print(f"\n{TOTAL_VOLUME:,} documents a month, {TOTAL_VOLUME * FIELDS_PER_DOCUMENT:,} fields a month")
print("\nwhat each segment mirrors:")
for _s in SEGMENTS:
    print(f"  {_s.name:24s} {_s.mirrors}")

## 3. Exercise 1 — what a plan costs, and what it buys

A **plan** gives every segment one tier: `plan[s]` is an index into `TIERS`. With n_s documents
a month in segment s, unit cost c(s, t) and field accuracy a(s, t) on tier t,

    cost per document(plan) = Σ_s n_s · c(s, plan[s]) / Σ_s n_s
    field accuracy(plan)    = Σ_s n_s · a(s, plan[s]) / Σ_s n_s

and the routing budget is the problem **maximise field accuracy(plan) subject to cost per
document(plan) ≤ B**. One choice from each group and one shared limit make it what operations
research calls a multiple-choice knapsack problem, which is classed as NP-hard (claims.yaml).
Both numbers are means over *documents*. A mean over segments treats a slice of a few hundred
documents like a slice of tens of thousands; the cell after this exercise measures what that
does to a bill.

<details><summary>💡 Hint 1 — what to think about</summary>

Two questions for every segment: how many documents does it carry, and what does one of them
cost and score on the tier the plan gives it? The per-page part of a price is where a long
document gets expensive, and `unit_cost` already knows about it. Before any arithmetic, decide
what a malformed plan should do — including an index Python would quietly accept.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Compare the plan's length with the number of segments, then check that every index lies
between zero and the last tier, rejecting negative ones yourself. Add up the documents and
refuse a total of zero. Then accumulate, segment by segment, volume times unit cost and volume
times the accuracy on the planned tier, and divide both sums by the total volume once, at the
end.

</details>

In [ ]:
def evaluate_plan(plan: Sequence[int], segments: Sequence[Segment],
                  tiers: Sequence[Tier]) -> PlanPoint:
    """Cost per document and field accuracy of one routing plan.

    `plan[s]` is the index into `tiers` of the tier that handles segment `s`. Both numbers are
    means over DOCUMENTS, not over segments, so every segment is weighted by its `volume`:

        cost_per_document = sum(volume_s * unit_cost(segment_s, tier)) / sum(volume_s)
        accuracy          = sum(volume_s * accuracy_s[tier])            / sum(volume_s)

    Raise ``ValueError`` when the plan does not give exactly one tier to every segment, when an
    index is not one of 0 .. len(tiers) - 1 (a negative index is a bug, not "the last tier"),
    or when the segments carry no documents at all.

    Example:
        >>> toy = (Segment("big", 3, 1, (0.5, 0.75), ""), Segment("small", 1, 1, (0.5, 0.75), ""))
        >>> two = (Tier("low", 1.0, 0.0, ""), Tier("high", 5.0, 0.0, ""))
        >>> evaluate_plan((1, 0), toy, two)       # (3 * 5.0 + 1 * 1.0) / 4, (3 * 0.75 + 1 * 0.5) / 4
        PlanPoint(cost_per_document=4.0, accuracy=0.6875)

    Returns:
        A ``PlanPoint(cost_per_document, accuracy)`` of plain floats.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
_TOY = (Segment("big", 3, 1, (0.5, 0.75), ""), Segment("small", 1, 1, (0.5, 0.75), ""))
_TWO = (Tier("low", 1.0, 0.0, ""), Tier("high", 5.0, 0.0, ""))


def _check_evaluate() -> None:
    got = evaluate_plan((1, 0), _TOY, _TWO)
    assert abs(got.cost_per_document - 4.0) < 1e-12 and abs(got.accuracy - 0.6875) < 1e-12, (
        f"evaluate_plan((1, 0), toy, two) gave {tuple(got)}, expected (4.0, 0.6875). (3.0, 0.625) "
        "is the mean over SEGMENTS: weight each segment by its volume and divide by total volume")
    long_doc = (Segment("long", 1, 4, (0.5, 0.75), ""),)
    by_page = (Tier("flat", 1.0, 0.0, ""), Tier("by page", 0.5, 0.25, ""))
    got = evaluate_plan((1,), long_doc, by_page)
    assert abs(got.cost_per_document - 1.5) < 1e-12, (
        f"a four-page document on a tier costing 0.5 plus 0.25 a page costs 1.5, got "
        f"{got.cost_per_document} — price each document with unit_cost(segment, tier)")
    for bad, why in (((1,), "the plan routes 1 segment and there are 2: compare len(plan) with "
                      "len(segments)"),
                     ((1, 0, 0), "the plan routes 3 segments and there are 2: compare len(plan) "
                      "with len(segments)"),
                     ((1, 2), "there are 2 tiers, so 2 is not a tier index: check every index "
                      "against len(tiers) yourself"),
                     ((0, -1), "-1 is a bug, not 'the last tier': reject negative indices "
                      "yourself, because Python will not")):
        try:
            evaluate_plan(bad, _TOY, _TWO)
        except ValueError:
            continue
        except IndexError:
            pass
        raise AssertionError(f"evaluate_plan({bad}, ...) must raise ValueError — {why}")
    print("exercise 1 looks right")


_try("exercise 1", _check_evaluate)

Run this for the three plans nobody should sign — everything on one tier — and for what the
mean over segments would have told you about one of them. Then read the two traps off the
fixture with your own function.

In [ ]:
def _listed(names: Sequence[str]) -> str:
    """["a"] -> "a"; ["a", "b", "c"] -> "a, b and c"."""
    names = list(names)
    return names[0] if len(names) == 1 else ", ".join(names[:-1]) + " and " + names[-1]


def _wrong_fields(point: PlanPoint) -> float:
    """Expected wrong fields a month over the whole traffic, for a plan landing at `point`."""
    return (1 - point.accuracy) * TOTAL_VOLUME * FIELDS_PER_DOCUMENT


def _show_single_tier_plans() -> None:
    n = len(SEGMENTS)
    print(f"{'everything on':15s}{'cost/doc':>10s}{'a month':>12s}{'accuracy':>10s}"
          f"{'wrong fields/month':>20s}")
    for k, tier in enumerate(TIERS):
        p = evaluate_plan((k,) * n, SEGMENTS, TIERS)
        print(f"{tier.name:15s}{p.cost_per_document:10.4f}{p.cost_per_document * TOTAL_VOLUME:12,.0f}"
              f"{p.accuracy:10.4f}{_wrong_fields(p):20,.0f}")

    everything = evaluate_plan((1,) * n, SEGMENTS, TIERS)
    naive_cost = sum(unit_cost(s, TIERS[1]) for s in SEGMENTS) / n
    naive_acc = sum(s.accuracy[1] for s in SEGMENTS) / n
    print(f"\neverything on the expensive tier, averaged over SEGMENTS instead of documents:")
    bill = (naive_cost - everything.cost_per_document) * TOTAL_VOLUME
    wrong = (everything.accuracy - naive_acc) * TOTAL_VOLUME * FIELDS_PER_DOCUMENT
    print(f"  cost per document {naive_cost:.4f} instead of {everything.cost_per_document:.4f} "
          f"— the monthly bill {'over' if bill > 0 else 'under'}stated by {abs(bill):,.0f}")
    print(f"  field accuracy    {naive_acc:.4f} instead of {everything.accuracy:.4f} "
          f"— {abs(wrong):,.0f} {'more' if wrong > 0 else 'fewer'} wrong fields a month reported "
          "than there are")

    cheap = evaluate_plan((0,) * n, SEGMENTS, TIERS)
    for name, tier in (("long supplier names", 1), ("poor scans", 1), ("poor scans", 2)):
        s = [seg.name for seg in SEGMENTS].index(name)
        moved = evaluate_plan(tuple(tier if i == s else 0 for i in range(n)), SEGMENTS, TIERS)
        print(f"\nfrom all-cheap, moving {name} to the {TIERS[tier].name} tier: cost per document "
              f"{moved.cost_per_document - cheap.cost_per_document:+.4f}, field accuracy "
              f"{moved.accuracy - cheap.accuracy:+.4f}")
    print("\nA move that costs more and scores the same, or worse, is one no budget should buy.")


_try("single-tier plans", _show_single_tier_plans, needs=("exercise 1",))

## 4. Exercise 2 — sweep every plan, then read off the best one under the cap

With three tiers and a handful of segments there are few enough plans to try every one, and
trying every one gives the only answer nobody can argue with. Build the sweep as arrays, so the
rest of the lesson can ask it questions cheaply, and then answer the first question: under a
cap B, which plan is the most accurate?

<details><summary>💡 Hint 1 — what to think about</summary>

The enumeration order is part of the answer, because the last tie-break is "lowest index".
For the cap, decide what happens to a plan that costs exactly the cap, and use the helper that
was written so that two programs agree about it. Then decide what wins when two plans that fit
are equally accurate — the docstring is specific, and "the first one numpy finds" is not it.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Take the product of the tier indices, repeated once per segment, in the library's own order,
and stack it into an integer array. Price each row with `evaluate_plan`, or build small tables
of every segment's unit cost and accuracy on every tier and index them with the whole plan
array at once. For the best plan: keep the indices `fits` accepts, raise if none is left,
narrow to those at the highest accuracy, narrow again to the cheapest of those, and return the
first index that remains, as a plain int.

</details>

In [ ]:
def sweep_plans(segments: Sequence[Segment], tiers: Sequence[Tier]) -> Sweep:
    """Every routing plan, with its cost per document and field accuracy.

    Plans come in ``itertools.product(range(len(tiers)), repeat=len(segments))`` order — the
    FIRST segment changes slowest — as an int array of shape
    ``(len(tiers) ** len(segments), len(segments))``. ``cost[k]`` and ``accuracy[k]`` are what
    ``evaluate_plan(plans[k], segments, tiers)`` returns, as float arrays. Loop over the plans or
    vectorise with numpy: the order and the numbers are what count.

    Example:
        >>> s = sweep_plans(toy, two)          # toy and two from exercise 1's docstring
        >>> s.plans.tolist()
        [[0, 0], [0, 1], [1, 0], [1, 1]]
        >>> s.cost.tolist(), s.accuracy.tolist()
        ([1.0, 2.0, 4.0, 5.0], [0.5, 0.5625, 0.6875, 0.75])

    Returns:
        ``Sweep(plans, cost, accuracy)``.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def best_within_budget(sweep: Sweep, budget: float) -> int:
    """Index into the sweep of the most accurate plan whose cost per document fits the cap.

    A plan fits when ``fits(cost, budget)`` is true: at or under the cap, with the float slack
    that helper allows. Among the plans that fit, take the highest accuracy; if several share
    it, the cheapest of those; if several share that too, the lowest index. Raise
    ``ValueError`` when no plan fits.

    Example:
        >>> s = sweep_plans(toy, two)
        >>> best_within_budget(s, 4.0)        # plan [1, 0] costs exactly 4.0, and that fits
        2
        >>> best_within_budget(s, 3.99)
        1

    Returns:
        A plain ``int``.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_sweep() -> None:
    s = sweep_plans(_TOY, _TWO)
    assert np.asarray(s.plans).tolist() == [[0, 0], [0, 1], [1, 0], [1, 1]], (
        f"plans came out as {np.asarray(s.plans).tolist()}; itertools.product order puts the "
        "FIRST segment slowest: [[0, 0], [0, 1], [1, 0], [1, 1]]")
    assert np.allclose(s.cost, [1.0, 2.0, 4.0, 5.0]) and np.allclose(
        s.accuracy, [0.5, 0.5625, 0.6875, 0.75]), (
        f"cost {np.asarray(s.cost).tolist()} / accuracy {np.asarray(s.accuracy).tolist()}: each "
        "row must be what evaluate_plan returns for that plan, volume-weighted")
    assert best_within_budget(s, 4.0) == 2, (
        "a plan that costs exactly the cap fits it: use fits(cost, budget), not cost < budget")
    assert best_within_budget(s, 3.99) == 1, (
        "at a cap of 3.99 the best plan that FITS is index 1 — the most accurate plan overall is "
        "not the answer, and neither is the cheapest")
    tie = sweep_plans((Segment("s", 1, 1, (0.5, 0.5), ""),),
                      (Tier("pricey", 4.0, 0.0, ""), Tier("thrifty", 1.0, 0.0, "")))
    assert best_within_budget(tie, 10.0) == 1, (
        "two plans fit with equal accuracy, and the second is cheaper: the tie goes to the "
        "cheaper plan, not to the first one np.argmax finds")
    try:
        best_within_budget(s, 0.5)
        raise AssertionError("no plan costs 0.5 or less, so best_within_budget must raise ValueError")
    except ValueError:
        pass
    print("exercise 2 looks right")


_try("exercise 2", _check_sweep)

Run this to sweep the real router at the cap finance set — and to time the sweep, because the
number of plans is the whole problem with it.

In [ ]:
def _print_routing(plan: Sequence[int], indent: str = "  ") -> None:
    """One line per tier: the segments the plan sends there."""
    for k, tier in enumerate(TIERS):
        names = [s.name for s, t in zip(SEGMENTS, plan) if t == k]
        print(f"{indent}{tier.name + ':':11s}{_listed(names) if names else '—'}")


def _show_sweep() -> None:
    t0 = time.perf_counter()
    sweep = sweep_plans(SEGMENTS, TIERS)
    secs = time.perf_counter() - t0
    n_plans, n_seg = np.asarray(sweep.plans).shape
    n_fit = int(np.count_nonzero(fits(sweep.cost, BUDGET_PER_DOCUMENT)))
    k = best_within_budget(sweep, BUDGET_PER_DOCUMENT)
    print(f"{n_plans:,} plans ({len(TIERS)} tiers to the power {n_seg}); {n_fit:,} of them fit "
          f"the cap of {BUDGET_PER_DOCUMENT:.4f} per document. The most accurate of those:")
    _print_routing(sweep.plans[k])
    print(f"  cost per document {sweep.cost[k]:.4f} · field accuracy {sweep.accuracy[k]:.4f} · "
          f"{sweep.cost[k] * TOTAL_VOLUME:,.0f} a month")
    per = secs / (n_plans * n_seg)
    print(f"\nthe sweep took {secs:.4f} s here, ≈ {per * 1e9:,.0f} ns per plan and segment. "
          "A rough extrapolation at that rate:")
    for m in (12, 16, 20):
        plans = len(TIERS) ** m
        print(f"  {m} segments: {plans:>15,} plans, ≈ {per * plans * m:,.1f} s")


_try("the sweep", _show_sweep, needs=("exercise 1", "exercise 2"))

## 5. Exercise 3 — the Pareto frontier

Most of the plans in the sweep are ones nobody should buy: some other plan costs no more and
gets at least as many fields right. What is left is the **Pareto frontier** — the menu finance
actually chooses from, and the shape in which a recent benchmark of document-extraction
systems reports cost against accuracy (claims.yaml). Every cap picks exactly one item off it.

The trap is the shape. Plans come in whole steps, so the frontier is a staircase, not a smooth
curve, and it is not the convex hull either: a plan can sit below the straight line joining two
of its neighbours and still be the best thing a cap between them can buy.

<details><summary>💡 Hint 1 — what to think about</summary>

Walk along the plans from cheapest to dearest. What must be true of a plan's accuracy, compared
with every plan you have already walked past, for it to be worth its extra cost? Then settle
the ties before they settle themselves: two plans at the same cost, two at the same accuracy,
and two that are identical.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Order the plans by cost ascending, breaking ties by accuracy descending and then by index
ascending — numpy's `lexsort` takes its keys last-key-first. Walk that order keeping the best
accuracy seen so far, and keep a plan only when its accuracy is strictly higher than that
best. What you kept is already in order of cost.

</details>

In [ ]:
def pareto_frontier(cost: np.ndarray, accuracy: np.ndarray) -> np.ndarray:
    """Indices of the plans that no other plan beats, in order of increasing cost.

    Plan k is on the frontier when no other plan costs no more AND is at least as accurate AND
    is strictly better on one of the two. So of two plans with equal accuracy only the cheaper
    survives, and of two with equal cost only the more accurate. Exact duplicates (equal cost
    and equal accuracy) keep only the lowest index. Along the returned order both cost and
    accuracy rise strictly. It is a staircase, NOT the convex hull.

    Example:
        >>> pareto_frontier(np.array([1.0, 2.0, 4.0, 5.0, 3.0]),
        ...                 np.array([0.5, 0.5625, 0.6875, 0.75, 0.55]))
        array([0, 1, 2, 3])

    Returns:
        An int numpy array of indices (empty input gives an empty array).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_frontier() -> None:
    got = np.asarray(pareto_frontier(np.array([1.0, 2.0, 4.0, 5.0, 3.0]),
                                     np.array([0.5, 0.5625, 0.6875, 0.75, 0.55]))).tolist()
    assert got == [0, 1, 2, 3], (
        f"got {got}, expected [0, 1, 2, 3]: plan 4 (cost 3.0, accuracy 0.55) is beaten by plan 1, "
        "which costs less and scores more, and the answer is ordered by cost")
    got = np.asarray(pareto_frontier(np.array([0.0, 1.0, 2.0]), np.array([0.0, 0.4, 1.0]))).tolist()
    assert got == [0, 1, 2], (
        f"got {got}: plan 1 sits below the straight line from plan 0 to plan 2 but nothing beats "
        "it — a cap of 1.0 buys it. The frontier is a staircase, not the convex hull")
    got = np.asarray(pareto_frontier(np.array([1.0, 2.0]), np.array([0.5, 0.5]))).tolist()
    assert got == [0], f"got {got}: equal accuracy at a higher cost is beaten; use > , not >="
    got = np.asarray(pareto_frontier(np.array([1.0, 1.0]), np.array([0.4, 0.5]))).tolist()
    assert got == [1], f"got {got}: at equal cost only the more accurate plan survives"
    got = np.asarray(pareto_frontier(np.array([2.0, 2.0]), np.array([0.7, 0.7]))).tolist()
    assert got == [0], f"got {got}: of two identical plans keep only the lower index"
    got = np.asarray(pareto_frontier(np.array([3.0, 1.0, 2.0]), np.array([0.9, 0.1, 0.5]))).tolist()
    assert got == [1, 2, 0], f"got {got}: return the indices in order of increasing COST"
    print("exercise 3 looks right")


_try("exercise 3", _check_frontier)

Run this for the picture: every plan as a grey dot, the frontier as a staircase, the cap as a
dashed line and the plan the cap buys as a star. The cost axis is logarithmic, because the
human tier costs orders of magnitude more than the machines.

In [ ]:
def _plot_frontier(path: Sequence[PlanPoint] = ()) -> None:
    sweep = sweep_plans(SEGMENTS, TIERS)
    front = pareto_frontier(sweep.cost, sweep.accuracy)
    k = best_within_budget(sweep, BUDGET_PER_DOCUMENT)
    fig, ax = plt.subplots(figsize=(10, 5.5))
    ax.scatter(sweep.cost, sweep.accuracy, s=4, color="0.75", label=f"all {len(sweep.cost):,} plans")
    ax.step(sweep.cost[front], sweep.accuracy[front], where="post", color="tab:blue", lw=1.2,
            label=f"Pareto frontier ({len(front)} plans)")
    ax.plot(sweep.cost[front], sweep.accuracy[front], "o", ms=3, color="tab:blue")
    if path:
        ax.plot([p.cost_per_document for p in path], [p.accuracy for p in path], ":s", ms=4,
                color="tab:orange", lw=1.0, label=f"greedy's path ({len(path)} plans)")
    ax.axvline(BUDGET_PER_DOCUMENT, color="black", ls="--", lw=0.8,
               label=f"cap {BUDGET_PER_DOCUMENT:.2f} per document")
    ax.plot([sweep.cost[k]], [sweep.accuracy[k]], "*", ms=14, color="tab:red",
            label="the plan the cap buys")
    ax.set_xscale("log")
    ax.set_xlabel("cost per document (illustrative placeholder prices, log scale)")
    ax.set_ylabel("field accuracy")
    ax.set_title("quality against cost per document", fontsize=10)
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    _show(fig)
    menu = sum(best_within_budget(sweep, sweep.cost[j]) == j for j in front)
    print(f"{len(front)} of {len(sweep.cost):,} plans are on the frontier. With its own cost as the "
          f"cap, the sweep picks that very plan for {menu} of the {len(front)}: the frontier is the "
          "list of every answer a cap can give.")


_try("frontier plot", _plot_frontier, needs=("exercise 1", "exercise 2", "exercise 3"))

## 6. Exercise 4 — the greedy marginal-yield rule

The sweep grows as three to the power of the number of segments; the extrapolation in section
4 says how soon that stops being affordable. The rule people reach for instead is greedy:
start every segment on its cheapest tier, then keep buying the move that adds the most accuracy
per unit of cost, for as long as a move still fits under the cap. It enumerates nothing. This
exercise implements it, and the cells after it measure exactly where it can be trusted.

<details><summary>💡 Hint 1 — what to think about</summary>

Three places this rule goes wrong in practice. A segment whose middle tier adds nothing must
still be able to reach the top tier. A move that does not fit is a reason to try the NEXT best
move, not to stop. And a move that costs more and scores less is never worth buying, however
much of the budget is left. Decide, too, what "yield" is per segment and why a segment's volume
does not belong in it.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Price every segment on every tier once with `unit_cost`. Start each segment on its cheapest
tier, breaking ties as the docstring says, and refuse a start that does not fit. Keep the
plan's total spend as a running sum. On every round, look at every segment and every tier that
is strictly dearer and strictly more accurate than where that segment is now, skip those whose
new total would not fit, and remember the one with the highest yield, keeping the first on a
tie. If there is none, stop; otherwise apply it, update the spend, record the move and go
round again.

</details>

In [ ]:
def greedy_allocation(segments: Sequence[Segment], tiers: Sequence[Tier],
                      budget: float) -> Greedy:
    """The greedy marginal-yield rule: buy accuracy where it is cheapest until nothing fits.

    1. START every segment on its cheapest tier by ``unit_cost``; a tie goes to the more
       accurate tier, then to the lower index. If that plan does not fit the budget
       (``fits``), raise ``ValueError``.
    2. A MOVE sends one segment from its current tier to ANY tier that is strictly dearer AND
       strictly more accurate for that segment — not only the next tier up — and is allowed only
       if the whole plan still fits the budget afterwards.
    3. A move's YIELD is the accuracy it adds per unit of cost it adds, for that segment:
       ``(a(s, new) - a(s, now)) / (c(s, new) - c(s, now))``. It is also the plan's field
       accuracy gained per unit of cost per document spent: the segment's volume cancels.
    4. Make the allowed move with the highest yield (ties: lower segment index, then lower tier
       index) and go back to 2. Stop when no move is allowed.

    Example:
        >>> greedy_allocation(toy, two, 4.0)      # both moves yield 0.0625; the tie goes to 'big'
        Greedy(plan=(1, 0), start=(0, 0), moves=((0, 1),))
        >>> greedy_allocation(toy, two, 3.0)      # 'big' no longer fits, so 'small' moves instead
        Greedy(plan=(0, 1), start=(0, 0), moves=((1, 1),))

    Returns:
        ``Greedy(plan, start, moves)``: the final plan and the starting plan as tuples of ints,
        and every move made, in order, as ``(segment index, new tier index)``.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_greedy() -> None:
    got = greedy_allocation(_TOY, _TWO, 100.0)
    assert [tuple(m) for m in got.moves] == [(0, 1), (1, 1)], (
        f"with room for everything got moves {[tuple(m) for m in got.moves]}, expected "
        "[(0, 1), (1, 1)]: both moves yield the same, so the tie goes to the lower segment index "
        "— keep the first best move you find, do not replace it on an equal yield")
    got = greedy_allocation(_TOY, _TWO, 4.0)
    assert tuple(got.plan) == (1, 0) and tuple(got.start) == (0, 0), (
        f"at a cap of 4.0 got plan {tuple(got.plan)} from {tuple(got.start)}, expected (1, 0) from "
        "(0, 0): moving 'big' leaves the plan costing exactly 4.0 PER DOCUMENT, and a plan at the "
        "cap fits — compare cost per document with the cap using fits(), not <")
    got = greedy_allocation(_TOY, _TWO, 3.0)
    assert tuple(got.plan) == (0, 1), (
        f"at a cap of 3.0 got {tuple(got.plan)}, expected (0, 1): moving 'big' does not fit, "
        "so try the next best move instead of stopping")
    jump = (Segment("s", 1, 1, (0.5, 0.5, 0.9), ""),)
    three = (Tier("a", 1.0, 0.0, ""), Tier("b", 2.0, 0.0, ""), Tier("c", 10.0, 0.0, ""))
    got = greedy_allocation(jump, three, 100.0)
    assert tuple(got.plan) == (2,) and [tuple(m) for m in got.moves] == [(0, 2)], (
        f"got {tuple(got.plan)} with moves {list(got.moves)}; the middle tier adds nothing here, "
        "so a move must be allowed straight to ANY dearer and more accurate tier, not only the "
        "next one up")
    worse = (Segment("s", 1, 1, (0.8, 0.7), ""),)
    got = greedy_allocation(worse, _TWO, 100.0)
    assert tuple(got.plan) == (0,), (
        f"got {tuple(got.plan)}: the dearer tier is LESS accurate for this segment, so moving "
        "there is never allowed, however much budget is left")
    big = (Segment("big", 4, 1, (0.5, 0.625), ""), Segment("small", 1, 1, (0.5, 0.75), ""))
    got = greedy_allocation(big, _TWO, 100.0)
    assert [tuple(m) for m in got.moves] == [(1, 1), (0, 1)], (
        f"moves {list(got.moves)}; 'small' yields 0.25 / 4 per unit of cost and 'big' 0.125 / 4, "
        "so 'small' moves first. Weighting the yield by the segment's volume gets the order "
        "wrong: the volume cancels")
    steep = (Segment("steep", 4, 1, (0.5, 0.625), ""), Segment("long", 1, 4, (0.5, 0.875), ""))
    got = greedy_allocation(steep, (Tier("flat", 1.0, 0.0, ""), Tier("by page", 0.0, 2.0, "")), 100.0)
    assert [tuple(m) for m in got.moves] == [(0, 1), (1, 1)], (
        f"moves {list(got.moves)}; 'steep' adds 0.125 for 1.0 and 'long' adds 0.375 for 7.0, so "
        "'steep' has the higher yield and moves first. Buying the largest GAIN first is not the "
        "marginal-yield rule")
    backwards = (Tier("pricey", 4.0, 0.0, ""), Tier("thrifty", 1.0, 0.0, ""))
    got = greedy_allocation((Segment("s", 1, 1, (0.5, 0.5), ""),), backwards, 100.0)
    assert tuple(got.start) == (1,), (
        f"start {tuple(got.start)}: tier 1 is the cheapest here, so the segment starts there — "
        "the cheapest tier is not always tier 0")
    try:
        greedy_allocation(_TOY, _TWO, 0.5)
        raise AssertionError("even the cheapest plan costs 1.0, so a cap of 0.5 must raise "
                             "ValueError rather than return a plan over the cap")
    except ValueError:
        pass
    try:
        got = greedy_allocation(_TOY, _TWO, 1.0)
    except ValueError:
        raise AssertionError("at a cap of 1.0 the starting plan costs exactly 1.0, and a plan at "
                             "the cap fits: check the start with fits(), not <") from None
    assert tuple(got.plan) == (0, 0) and not got.moves, (
        f"at a cap of 1.0 got {tuple(got.plan)}: the start costs exactly the cap, so no move fits")
    print("exercise 4 looks right")


_try("exercise 4", _check_greedy)

Run this to watch the rule spend an unlimited budget. Each row is one move; the running cost
and accuracy after it are one point of greedy's **path**. Look at where the poor scans go, and
where the long supplier names do not.

In [ ]:
def greedy_path(segments: Sequence[Segment], tiers: Sequence[Tier]) -> list[tuple[int, ...]]:
    """The plans greedy passes through with no cap, the starting plan first."""
    run = greedy_allocation(segments, tiers, math.inf)
    plans, plan = [tuple(run.start)], list(run.start)
    for s, t in run.moves:
        plan[s] = t
        plans.append(tuple(plan))
    return plans


def _show_greedy_path() -> None:
    run = greedy_allocation(SEGMENTS, TIERS, math.inf)
    plan = list(run.start)
    here = evaluate_plan(plan, SEGMENTS, TIERS)
    yields = []
    print(f"{'move':50s}{'yield':>9s}{'cost/doc':>10s}{'accuracy':>10s}")
    print(f"{'(start: every segment on its cheapest tier)':50s}{'':9s}"
          f"{here.cost_per_document:10.4f}{here.accuracy:10.4f}")
    for s, t in run.moves:
        seg = SEGMENTS[s]
        y = (seg.accuracy[t] - seg.accuracy[plan[s]]) / (
            unit_cost(seg, TIERS[t]) - unit_cost(seg, TIERS[plan[s]]))
        label = f"{seg.name}: {TIERS[plan[s]].name} -> {TIERS[t].name}"
        yields.append(y)
        plan[s] = t
        here = evaluate_plan(plan, SEGMENTS, TIERS)
        print(f"{label:50s}{y:9.4f}{here.cost_per_document:10.4f}{here.accuracy:10.4f}")
    visits = {s: [TIERS[run.start[s]].name] + [TIERS[t].name for m, t in run.moves if m == s]
              for s in range(len(SEGMENTS))}
    falling = all(a >= b for a, b in zip(yields, yields[1:]))
    print(f"\nyields {'never rise' if falling else 'do NOT always fall'} down the list: greedy "
          "buys the cheapest accuracy first.")
    for name in ("poor scans", "long supplier names"):
        s = [seg.name for seg in SEGMENTS].index(name)
        print(f"{name}: " + " -> ".join(visits[s]))


_try("greedy path", _show_greedy_path, needs=("exercise 1", "exercise 4"))

Now the measurement this section exists for: *where* is greedy right? Run this to compare it
with the sweep at a spread of caps, at greedy's own breakpoints (the costs of the plans on its
path), and against a price per wrong field instead of a cap. Then the same comparison on random
segment mixes drawn on these tiers from a fixed seed, so the answer is not a property of one
fixture. The bound it checks is the classical one for this problem (claims.yaml): a shortfall
can never exceed the value of the one move greedy could not afford.

In [ ]:
def random_mix(rng: random.Random, n_segments: int) -> tuple[Segment, ...]:
    """A random segment mix on this lesson's tiers. Deterministic for a given generator state."""
    out = []
    for i in range(n_segments):
        cheap = rng.uniform(0.40, 0.90)
        expensive = min(0.99, max(0.0, cheap + rng.uniform(-0.05, 0.40)))
        human = min(0.995, max(cheap, expensive) + rng.uniform(0.0, 0.10))
        out.append(Segment(f"random {i}", rng.randint(100, 20_000), rng.choice((1, 1, 1, 2, 3, 12)),
                           (cheap, expensive, human), "random mix"))
    return tuple(out)


def _compare(segments: Sequence[Segment], caps: Sequence[float]) -> list[float]:
    """Sweep accuracy minus greedy accuracy at every cap."""
    sweep = sweep_plans(segments, TIERS)
    gaps = []
    for cap in caps:
        best = sweep.accuracy[best_within_budget(sweep, cap)]
        gaps.append(float(best - evaluate_plan(greedy_allocation(segments, TIERS, cap).plan,
                                               segments, TIERS).accuracy))
    return gaps


def _next_move_value(path_points: Sequence[PlanPoint], cap: float) -> float:
    """Accuracy the first move on greedy's path that does not fit under `cap` would have added."""
    k = max(i for i, p in enumerate(path_points) if fits(p.cost_per_document, cap))
    return path_points[k + 1].accuracy - path_points[k].accuracy if k + 1 < len(path_points) else 0.0


def _show_where_greedy_is_optimal() -> None:
    sweep = sweep_plans(SEGMENTS, TIERS)
    path = [evaluate_plan(p, SEGMENTS, TIERS) for p in greedy_path(SEGMENTS, TIERS)]
    lo, hi = float(sweep.cost.min()), float(sweep.cost.max())
    caps = [round(c, 6) for c in np.geomspace(lo * 1.01, hi, 241)]
    gaps = _compare(SEGMENTS, caps)
    worst = int(np.argmax(gaps))
    same = sum(g <= 1e-12 for g in gaps)
    print(f"at {len(caps)} caps from {caps[0]:.4f} to {caps[-1]:.4f} per document, greedy matched "
          f"the sweep's accuracy at {same} and fell short at {len(caps) - same}.")
    print(f"worst shortfall: {gaps[worst]:.4f} field accuracy at a cap of {caps[worst]:.4f}, which is "
          f"{gaps[worst] * TOTAL_VOLUME * FIELDS_PER_DOCUMENT:,.0f} more wrong fields a month.")
    bps = [p.cost_per_document for p in path]
    at_bp = _compare(SEGMENTS, bps)
    print(f"at its own {len(bps)} breakpoints greedy matched the sweep {sum(g <= 1e-12 for g in at_bp)}"
          f" times out of {len(bps)}.")
    bounded = sum(g <= _next_move_value(path, c) + 1e-12 for g, c in zip(gaps, caps))
    print(f"the shortfall was no larger than the move greedy could not afford at {bounded} of "
          f"{len(caps)} caps.")

    wrong_all = (1 - sweep.accuracy) * FIELDS_PER_DOCUMENT
    wrong_path = np.array([(1 - p.accuracy) * FIELDS_PER_DOCUMENT for p in path])
    cost_path = np.array(bps)
    prices = np.geomspace(0.01, 1000.0, 61)
    exact = sum(abs((sweep.cost + p * wrong_all).min() - (cost_path + p * wrong_path).min()) <= 1e-9
                for p in prices)
    print(f"priced instead of capped — minimise spend plus a price per wrong field — greedy's "
          f"path holds the best plan at {exact} of {len(prices)} prices from {prices[0]:.2f} to "
          f"{prices[-1]:,.0f}.")

    rng = random.Random(SEED)
    trials, beaten, shortfalls, bound_ok = 150, 0, [], 0
    for _ in range(trials):
        mix = random_mix(rng, 6)
        mix_sweep = sweep_plans(mix, TIERS)
        mlo, mhi = float(mix_sweep.cost.min()), float(mix_sweep.cost.max())
        cap = math.exp(rng.uniform(math.log(mlo), math.log(mhi)))
        gap = _compare(mix, [cap])[0]
        mix_path = [evaluate_plan(p, mix, TIERS) for p in greedy_path(mix, TIERS)]
        bound_ok += gap <= _next_move_value(mix_path, cap) + 1e-12
        if gap > 1e-12:
            beaten += 1
            shortfalls.append(gap)
    print(f"\n{trials} random six-segment mixes on these tiers, each at a random cap (seed {SEED}): "
          f"greedy matched the sweep on {trials - beaten}; on the other {beaten} it fell short by "
          f"at most {max(shortfalls, default=0.0):.4f} (median "
          f"{float(np.median(shortfalls)) if shortfalls else 0.0:.4f}). The bound held on "
          f"{bound_ok} of {trials}.")


_try("where greedy is optimal", _show_where_greedy_is_optimal,
     needs=("exercise 1", "exercise 2", "exercise 4"))

The frontier again, with greedy's path drawn over it. Every plan on the path is on the frontier;
the frontier plans between two path points are the ones only a cap can ask for — and the only
ones greedy can miss.

In [ ]:
def _plot_frontier_and_path() -> None:
    _plot_frontier([evaluate_plan(p, SEGMENTS, TIERS) for p in greedy_path(SEGMENTS, TIERS)])


_try("frontier and path", _plot_frontier_and_path,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 7. Exercise 5 — build the case where greedy loses

The measurements say greedy is beaten only between its own breakpoints: at caps where the next
move on its path does not fit, so it spends what is left on smaller moves and never undoes the
ones it already made. Make that happen on purpose — on this router's own `TIERS`, with segments
you design — and make the loss big enough to see.

<details><summary>💡 Hint 1 — what to think about</summary>

Greedy never undoes a move. So look for a move it will buy first because it is the steepest,
which then leaves too little room for a move that is a little less steep but worth far more in
total. How does a segment's volume change what a move is worth to the plan without changing its
yield? Where must the cap sit relative to what each move costs?

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Use two segments. Give a small one a very steep move and a large one a slightly shallower
move to the same tier, so that greedy takes the small one first. Put the cap at what the plan
with ONLY the large segment moved costs — your own `evaluate_plan` will tell you — so that the
large move fits on its own but not after the small one. Check the gap with your sweep and your
greedy before you hand it in.

</details>

In [ ]:
def greedy_counterexample() -> tuple[tuple[Segment, ...], float]:
    """A segment mix and a cap, on THIS lesson's ``TIERS``, where greedy loses to the sweep.

    Build it yourself. The rubric checks it with a reference greedy and a reference sweep, not
    with yours, so it must be a genuine counterexample:

    * between 2 and 6 segments, each with a positive whole ``volume`` and ``pages``, and an
      ``accuracy`` holding one value in [0, 1] for each of the three ``TIERS``;
    * a cap at or above the cost of greedy's starting plan, so that greedy can start;
    * the sweep's best plan within that cap at least 0.001 more accurate than greedy's plan.

    The shape of an answer (this particular one is NOT a counterexample: greedy finds the
    optimum on it):
        >>> mix = (Segment("a", 10, 1, (0.5, 0.9, 0.95), ""), Segment("b", 10, 1, (0.5, 0.6, 0.95), ""))
        >>> return mix, 0.05

    Returns:
        ``(segments, budget)``: a tuple of ``Segment`` and a float cap on cost per document.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_counterexample() -> None:
    mix, cap = greedy_counterexample()
    assert 2 <= len(mix) <= 6, f"use between 2 and 6 segments, not {len(mix)}"
    for s in mix:
        assert int(s.volume) == s.volume > 0 and int(s.pages) == s.pages > 0, (
            f"{s.name!r}: volume and pages must be positive whole numbers")
        assert len(s.accuracy) == len(TIERS) and all(0.0 <= a <= 1.0 for a in s.accuracy), (
            f"{s.name!r}: give one accuracy in [0, 1] for each of the {len(TIERS)} TIERS")
    sweep = sweep_plans(mix, TIERS)
    best = float(sweep.accuracy[best_within_budget(sweep, cap)])
    got = evaluate_plan(greedy_allocation(mix, TIERS, cap).plan, mix, TIERS).accuracy
    assert best - got >= 0.001, (
        f"the sweep reaches {best:.4f} and greedy {got:.4f} at your cap of {cap}: not a "
        "counterexample yet. The cap has to let the big move fit alone but not after the small, "
        "steep one greedy buys first")
    print("exercise 5 looks right")


_try("exercise 5", _check_counterexample, needs=("exercise 1", "exercise 2", "exercise 4"))

Run this to see your counterexample from the inside: what greedy bought, in what order, and
what the cap would have bought instead.

In [ ]:
def _show_counterexample() -> None:
    mix, cap = greedy_counterexample()
    sweep = sweep_plans(mix, TIERS)
    k = best_within_budget(sweep, cap)
    run = greedy_allocation(mix, TIERS, cap)
    g = evaluate_plan(run.plan, mix, TIERS)
    print(f"cap {cap:.6f} per document, {len(mix)} segments")
    for s in mix:
        print(f"  {s.name:28s} {s.volume:7,d} docs  accuracy "
              + " / ".join(f"{a:.3f}" for a in s.accuracy))
    print("greedy's moves: " + (", ".join(f"{mix[s].name} -> {TIERS[t].name}"
                                          for s, t in run.moves) or "none"))
    print(f"greedy:  {tuple(run.plan)}  cost {g.cost_per_document:.6f}  accuracy {g.accuracy:.4f}"
          f"  (leaves {cap - g.cost_per_document:.6f} of the cap unspent)")
    print(f"sweep:   {tuple(int(t) for t in sweep.plans[k])}  cost {sweep.cost[k]:.6f}  accuracy "
          f"{sweep.accuracy[k]:.4f}")


_try("your counterexample", _show_counterexample, needs=("exercise 1", "exercise 2", "exercise 4",
                                                          "exercise 5"))

## 8. Exercise 6 — the recommendation a finance partner signs

A finance partner does not sign a frontier. They sign a sentence: what a document will cost,
what the month will cost, how many fields will still be wrong, and what it would cost to do
better. Every figure in that sentence has to come from the numbers, because a figure somebody
typed is one nobody can reproduce. Your job is the numbers; `render_recommendation`, given
below the stub, turns them into the paragraph and adds nothing of its own.

The figure that matters most is the last one: the **price per wrong field avoided** of the next
step up the frontier. If a wrong field costs the business more than that, raise the cap.

<details><summary>💡 Hint 1 — what to think about</summary>

The chosen plan is the sweep's, not greedy's, because at this size the sweep is affordable and
section 6 measured what greedy leaves on the table. "The next step" is the next plan on the
FRONTIER, not the next plan by cost: most dearer plans are worse. Accuracy is per field, so
ask how many fields a month there are before counting the wrong ones. And decide what the top
of the frontier should say.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Sweep, take the best plan within the cap, and evaluate it. Monthly spend is cost per document
times the documents a month; wrong fields a month are one minus accuracy, times documents,
times fields per document — the same for greedy's plan at the same cap. Find the chosen plan's
position on the frontier; if another frontier plan follows it, that is the next step, and its
price is the extra monthly spend divided by the wrong fields it removes. If none follows, the
next plan, its point and its price are all None.

</details>

In [ ]:
def recommend(segments: Sequence[Segment], tiers: Sequence[Tier],
              budget: float) -> Recommendation:
    """Every figure a finance partner needs to sign a routing budget, computed, never typed.

    * ``plan`` and ``point``: the sweep's best plan within ``budget`` (``best_within_budget``),
      as a tuple of ints, and its ``evaluate_plan``. Not greedy's plan.
    * ``plans_checked``: how many plans the sweep compared — all of them.
    * ``monthly_spend``: cost per document x the documents a month (the segments' volumes).
    * ``wrong_fields_per_month``: (1 - accuracy) x documents a month x ``FIELDS_PER_DOCUMENT``.
    * ``greedy_wrong_fields_per_month``: the same for ``greedy_allocation``'s plan at the same
      budget, so the paragraph can say what the shortcut would have cost.
    * ``next_plan`` and ``next_point``: the plan that follows the chosen one on the Pareto
      frontier — the cheapest frontier plan more accurate than it — or ``None`` for both when
      the chosen plan is already the most accurate there is.
    * ``price_per_wrong_field_avoided``: (next monthly spend - monthly spend) / (wrong fields a
      month - next wrong fields a month), or ``None`` when there is no next plan.
    * ``budget``: the cap, echoed back.

    Example:
        >>> rec = recommend(toy, two, 4.0)
        >>> rec.plan, rec.monthly_spend, rec.next_plan, rec.price_per_wrong_field_avoided
        ((1, 0), 16.0, (1, 1), 2.6666666666666665)

    Returns:
        A ``Recommendation``.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def render_recommendation(rec: Recommendation, segments: Sequence[Segment],
                          tiers: Sequence[Tier]) -> str:
    """The paragraph, formatted from `rec` and the segment names, and from nothing else."""
    docs = sum(s.volume for s in segments)
    routes = []
    for k, tier in enumerate(tiers):
        names = [s.name for s, t in zip(segments, rec.plan) if t == k]
        routes.append(f"{_listed(names) if names else 'nothing'} to the {tier.name} tier")
    extra = rec.greedy_wrong_fields_per_month - rec.wrong_fields_per_month
    shortcut = (f"the greedy marginal-yield rule, held to the same cap, would leave {extra:,.0f} "
                "more fields wrong a month" if extra >= 0.5 else
                "the greedy marginal-yield rule, held to the same cap, does as well")
    text = (f"Fund document extraction at a cap of {rec.budget:.4f} per document and route "
            f"{'; '.join(routes)}. The plan costs {rec.point.cost_per_document:.4f} per document, "
            f"{rec.monthly_spend:,.0f} a month for {docs:,} documents, and leaves an expected "
            f"{rec.wrong_fields_per_month:,.0f} of {docs * FIELDS_PER_DOCUMENT:,} fields wrong a "
            f"month (field accuracy {rec.point.accuracy:.4f}). Of all {rec.plans_checked:,} "
            f"possible plans it is the most accurate that fits the cap, found by checking every "
            f"one; {shortcut}. ")
    if rec.next_plan is None:
        text += "No plan is more accurate than this one, so raising the cap buys nothing. "
    else:
        next_wrong = (1 - rec.next_point.accuracy) * docs * FIELDS_PER_DOCUMENT
        text += (f"The next step up the frontier costs {rec.next_point.cost_per_document:.4f} per "
                 f"document and avoids {rec.wrong_fields_per_month - next_wrong:,.0f} more wrong "
                 f"fields a month, at {rec.price_per_wrong_field_avoided:,.2f} per wrong field "
                 "avoided: raise the cap only if a wrong field costs the business more than that. ")
    return text + ("Every price behind these figures is an illustrative placeholder, not a quote "
                   "from any real vendor or payroll, and must be replaced with your own before "
                   "this is signed.")


def _check_recommend() -> None:
    rec = recommend(_TOY, _TWO, 4.0)
    assert tuple(rec.plan) == (1, 0) and rec.plans_checked == 4, (
        f"plan {tuple(rec.plan)} from {rec.plans_checked} plans: the sweep's best plan within the "
        "cap, from all of them")
    assert abs(rec.monthly_spend - 16.0) < 1e-9, (
        f"monthly spend {rec.monthly_spend}: cost per document 4.0 x 4 documents a month is 16.0")
    assert abs(rec.wrong_fields_per_month - 0.3125 * 4 * FIELDS_PER_DOCUMENT) < 1e-9, (
        f"wrong fields a month {rec.wrong_fields_per_month}: (1 - accuracy) x documents x "
        "FIELDS_PER_DOCUMENT — accuracy is a share of FIELDS, not of documents")
    assert rec.next_plan is not None and tuple(rec.next_plan) == (1, 1), (
        f"next plan {rec.next_plan}: the plan after the chosen one on the Pareto frontier is (1, 1)")
    assert rec.next_point is not None and abs(rec.next_point.cost_per_document - 5.0) < 1e-9 \
        and abs(rec.next_point.accuracy - 0.75) < 1e-9, (
            f"next point {rec.next_point}: it is evaluate_plan of the next plan, (5.0, 0.75) — the "
            "paragraph quotes its cost per document")
    assert abs(rec.price_per_wrong_field_avoided - 4.0 / 1.5) < 1e-9, (
        f"price {rec.price_per_wrong_field_avoided}: the next plan costs 4.0 more a month and "
        "leaves 1.5 fewer fields wrong a month, so 2.667 per wrong field avoided. 16.0 means the "
        "wrong fields were counted per document, not per field")
    one = (Segment("s", 4, 1, (0.5, 0.25, 0.75), ""),)
    three = (Tier("a", 1.0, 0.0, ""), Tier("b", 2.0, 0.0, ""), Tier("c", 3.0, 0.0, ""))
    got = recommend(one, three, 1.5)
    assert got.next_plan is not None and tuple(got.next_plan) == (2,), (
        f"next plan {got.next_plan}: (1,) is the next plan by COST but it is less accurate than "
        "the chosen (0,); the next step is the next plan on the FRONTIER, (2,)")
    top = recommend(_TOY, _TWO, 100.0)
    assert top.next_plan is None and top.next_point is None and \
        top.price_per_wrong_field_avoided is None, (
            "at a cap that buys the most accurate plan there is no next step: next_plan, "
            "next_point and the price must all be None")
    sweep = sweep_plans(SEGMENTS, TIERS)
    rec = recommend(SEGMENTS, TIERS, BUDGET_PER_DOCUMENT)
    want = tuple(int(t) for t in sweep.plans[best_within_budget(sweep, BUDGET_PER_DOCUMENT)])
    assert tuple(rec.plan) == want, (
        f"on the real router the chosen plan must be the sweep's {want}, got {tuple(rec.plan)} — "
        "greedy's plan is the shortcut the paragraph compares against, not the recommendation")
    g = evaluate_plan(greedy_allocation(SEGMENTS, TIERS, BUDGET_PER_DOCUMENT).plan, SEGMENTS, TIERS)
    assert abs(rec.greedy_wrong_fields_per_month - _wrong_fields(g)) < 1e-6, (
        f"greedy_wrong_fields_per_month {rec.greedy_wrong_fields_per_month:,.1f}, expected "
        f"{_wrong_fields(g):,.1f}: count the wrong fields of GREEDY's plan at the same cap")
    print("exercise 6 looks right")


_try("exercise 6", _check_recommend,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

Run this for the artefact: the routing budget, the recommendation it implies, and a trace of
every figure in the paragraph back to a computed value — the check a validator would make.

In [ ]:
def _figures_in(text: str) -> list[str]:
    """Every number printed in a piece of prose, as it is printed."""
    return re.findall(r"\d[\d,]*(?:\.\d+)?", text)


def _show_recommendation() -> None:
    rec = recommend(SEGMENTS, TIERS, BUDGET_PER_DOCUMENT)
    print(f"routing budget: cap {rec.budget:.4f} per document, {TOTAL_VOLUME:,} documents a month")
    _print_routing(rec.plan)
    paragraph = render_recommendation(rec, SEGMENTS, TIERS)
    print("\n" + textwrap.fill(paragraph, width=94))
    docs = TOTAL_VOLUME
    values = [rec.budget, rec.point.cost_per_document, rec.monthly_spend, docs,
              rec.wrong_fields_per_month, docs * FIELDS_PER_DOCUMENT, rec.point.accuracy,
              rec.plans_checked, rec.greedy_wrong_fields_per_month - rec.wrong_fields_per_month]
    if rec.next_plan is not None:
        values += [rec.next_point.cost_per_document, rec.price_per_wrong_field_avoided,
                   rec.wrong_fields_per_month
                   - (1 - rec.next_point.accuracy) * docs * FIELDS_PER_DOCUMENT]
    printed = {f"{v:,.0f}" for v in values} | {f"{v:.4f}" for v in values} | \
        {f"{v:,.2f}" for v in values}
    figures = _figures_in(paragraph)
    traced = [f for f in figures if f in printed]
    print(f"\n{len(figures)} figures in the paragraph; {len(traced)} of them trace to a value "
          "computed in this notebook.")


_try("the recommendation", _show_recommendation,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 6"))

## 9. Common mistakes

- **Averaging over segments.** Section 3 measured what the mean over segments does to a bill.
  Cost per document and accuracy are means over documents.
- **Buying a dearer tier because it is dearer.** On the long supplier names the expensive
  tier scores less; on the poor scans it scores the same. Neither move belongs in any plan.
- **Only ever stepping to the next tier up.** On the poor scans the middle tier adds nothing,
  so a rule that must pass through it never reaches the reviewer who does.
- **Stopping at the first move that does not fit.** The next best move may fit.
- **Trusting greedy between its breakpoints.** Section 6 measured where it falls short; when
  the sweep is affordable, recommend the sweep's plan and quote greedy's shortfall beside it.
- **Reading the convex hull as the frontier.** The hull is greedy's path. The staircase holds
  the plans only a cap can buy.
- **Quoting a price per document point instead of per wrong field.** Accuracy is per field;
  the price a finance partner compares with the cost of an error is per wrong field.
- **Treating the placeholders as prices.** They move the answer. Run the next cell.

In [ ]:
def _show_what_one_placeholder_moves() -> None:
    SECONDS_PER_REVIEWED_CELL = 40.0   # module 1's placeholder, under module 1's own name
    module_1_pace = 3600.0 / SECONDS_PER_REVIEWED_CELL   # fields an hour at that pace
    plans = []
    for label, tiers in (("module 6's measured pace", TIERS),
                         ("module 1's placeholder pace", build_tiers(module_1_pace))):
        rec = recommend(SEGMENTS, tiers, BUDGET_PER_DOCUMENT)
        plans.append(tuple(rec.plan))
        human = [s.name for s, t in zip(SEGMENTS, rec.plan) if t == 2]
        print(f"{label}: a human-reviewed document costs {tiers[2].per_document:.4f}; at the same "
              f"cap the plan costs {rec.point.cost_per_document:.4f} per document, field accuracy "
              f"{rec.point.accuracy:.4f}, and sends to human review: "
              f"{_listed(human) if human else 'nothing'}")
    moved = sum(a != b for a, b in zip(*plans))
    print(f"\nOne placeholder changed; {moved} of {len(SEGMENTS)} segments changed tier. Time your "
          "own reviewers before anyone signs.")


_try("one placeholder", _show_what_one_placeholder_moves,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 6"))

## 10. Self-check

1. The sweep's plan at your cap costs less than the cap. The unspent headroom means:
   - (a) the sweep has a bug: the best plan always spends the whole budget
   - (b) the cap should be cut to the plan's cost before anything else is decided
   - (c) no dearer plan that still fits is more accurate: plans come in whole steps, so the
         best one rarely lands exactly on the cap

2. The expensive tier is less accurate than the cheap one on the long supplier names. At a
   generous cap, the sweep will:
   - (a) never send that segment to the expensive tier, because a dearer and worse option is
         beaten by the cheap one — though it may send it to the reviewer
   - (b) send it to the expensive tier, because a larger cap should buy the dearer tier
   - (c) fail, because the sweep assumes accuracy rises with cost

3. At a cap that falls between two of greedy's breakpoints, greedy scores below the sweep.
   The cause is:
   - (a) a rounding error in the yields
   - (b) greedy bought a small, steep move first, and what was left could not pay for a
         larger move worth more in total — and greedy never undoes a move
   - (c) the sweep counted plans that cost more than the cap

4. Finance gives you a price per wrong field instead of a cap, and asks for the plan that
   minimises spend plus priced errors. On this lesson's measurements:
   - (a) only the sweep can answer, because greedy fails between its breakpoints
   - (b) the answer is always the most accurate plan
   - (c) greedy's path held the best plan at every price tried, so the cheap rule answers
         this question exactly

5. The next step up the frontier costs 5.00 per wrong field avoided, and the business loses
   about 2.00 per wrong field that reaches the ledger. The paragraph tells finance to:
   - (a) keep the cap: the next step costs more per error avoided than an error costs
   - (b) raise the cap: more accuracy is always worth buying
   - (c) cut the cap to the greedy plan's cost

Answers come with this lesson's worked solution when you enrol on Synapsa.

One last cell: the routing card. The frontier around the chosen plan, each step with its price
per wrong field avoided — the menu the recommendation was chosen from, every figure computed.

In [ ]:
def _show_routing_card() -> None:
    sweep = sweep_plans(SEGMENTS, TIERS)
    front = [int(j) for j in pareto_frontier(sweep.cost, sweep.accuracy)]
    k = best_within_budget(sweep, BUDGET_PER_DOCUMENT)
    at = front.index(k)
    print(f"{'':3s}{'cost/doc':>10s}{'a month':>10s}{'accuracy':>10s}{'wrong fields/month':>20s}"
          f"{'price of the step up to this row':>34s}")
    prices = []
    for i in range(max(0, at - 4), min(len(front), at + 5)):
        j = front[i]
        wrong = (1 - sweep.accuracy[j]) * TOTAL_VOLUME * FIELDS_PER_DOCUMENT
        if i:
            p = front[i - 1]
            step = (sweep.cost[j] - sweep.cost[p]) / ((sweep.accuracy[j] - sweep.accuracy[p])
                                                      * FIELDS_PER_DOCUMENT)
            price = f"{step:34,.2f}"
            prices.append(step)
        else:
            price = f"{'—':>34s}"
        mark = " ->" if j == k else "   "
        print(f"{mark}{sweep.cost[j]:10.4f}{sweep.cost[j] * TOTAL_VOLUME:10,.0f}"
              f"{sweep.accuracy[j]:10.4f}{wrong:20,.0f}{price}")
    dear = sum(a > b for a, b in zip(prices, prices[1:]))
    print(f"\n-> marks the plan a cap of {BUDGET_PER_DOCUMENT:.4f} buys; {len(front)} frontier "
          f"plans in all. In {dear} of the {max(len(prices) - 1, 0)} pairs of consecutive steps "
          "shown, a step\ncharges more per wrong field avoided than the step after it. The "
          "staircase is not convex, so a cap\nthat stops right after a dear step has paid more "
          "per wrong field than the next step would charge.")


_try("routing card", _show_routing_card, needs=("exercise 2", "exercise 3"))

## What you built, and where it goes next

You now own the cost model of the programme's pipeline: a volume-weighted price for any
routing plan, an exhaustive sweep and the Pareto frontier it draws, a greedy rule whose
failures you have measured and constructed rather than been warned about, and a recommendation
whose every figure traces back to a computation. The capstone asks you to put this card, the
drift monitor and the review policy in front of a validator, with every price replaced by one
of your own.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")


# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in (
                ("exercise 1", _check_evaluate, ()),
                ("exercise 2", _check_sweep, ()),
                ("exercise 3", _check_frontier, ()),
                ("exercise 4", _check_greedy, ()),
                ("exercise 5", _check_counterexample, ("exercise 1", "exercise 2", "exercise 4")),
                ("exercise 6", _check_recommend,
                 ("exercise 1", "exercise 2", "exercise 3", "exercise 4"))):
            _try(_name, _check, _needs)
    _progress_board()
    print(f"\nlesson wall time so far: {time.perf_counter() - _LESSON_T0:.1f} s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.